In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import os
import zipfile
from tqdm import tqdm


In [ ]:
# - CPUS - 

url = "https://www.tomshardware.com/reviews/cpu-hierarchy,4312.html"
headers = {'User-Agent': 'Mozilla/5.0'}  # Helps avoid potential blocking
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# Find the main table – it's usually the largest one or has a specific class; fallback to first table
table = soup.find('table')
if not table:
    raise ValueError("No table found – page structure may have changed significantly.")

rows = table.find_all('tr')
data = []

for row in rows[1:]:  # Skip header row
    cols = row.find_all('td')
    if len(cols) < 7:  # Skip incomplete or header-like rows
        continue
    
    # Column 0: Product name (may include MSRP/notes)
    product_full = cols[0].text.strip()
    # Extract clean CPU name (remove notes like "CU-8200 | DDR5-7200 ($589)")
    product = product_full.split('(')[0].split('|')[0].strip()
    
    # Column 1: Lowest price
    price = cols[1].text.strip()
    
    # Column 2: 1080p Gaming Score – clean robustly
    gaming_score_str = cols[2].text.strip()
    # Handle possible formats: "74.17%", "74.17% | 72.45%", or with extra spaces
    gaming_score_str = gaming_score_str.split('|')[0].strip()  # Take first part if split
    gaming_score_str = gaming_score_str.rstrip('%').strip()
    try:
        gaming_score = float(gaming_score_str)
    except ValueError:
        print(f"Skipping row due to invalid score: {gaming_score_str} in {product}")
        continue
    
    arch = cols[3].text.strip()
    cores_threads = cols[4].text.strip()
    clocks = cols[5].text.strip()
    tdp = cols[6].text.strip()
    
    # Simple manufacturer detection
    manufacturer = 'AMD' if 'Ryzen' in product else 'Intel' if 'Core' in product else 'Unknown'
    
    data.append({
        'cpu_name': product,
        'manufacturer': manufacturer,
        'gaming_score': gaming_score,
        'architecture': arch,
        'cores_threads': cores_threads,
        'base_boost_ghz': clocks,
        'tdp': tdp,
        'price_current': price,
        'product_full': product_full
    })

df_cpu = pd.DataFrame(data)
df_cpu['perf_score'] = df_cpu['gaming_score']  # Already on 0-100 relative scale
df_cpu = df_cpu.sort_values('perf_score', ascending=False).reset_index(drop=True)

df_cpu.to_csv('cpu_benchmarks_2026.csv', index=False)
print(f"Successfully scraped {len(df_cpu)} CPUs")
print(df_cpu[['cpu_name', 'perf_score', 'architecture']].head(10))

In [5]:
# - GPUS -

url = "https://www.tomshardware.com/reviews/gpu-hierarchy,4388.html"
headers = {'User-Agent': 'Mozilla/5.0'}  # Recommended for reliability
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# Find all tables and select the first one (Rasterization rankings – the main gaming table)
tables = soup.find_all('table')
if len(tables) < 1:
    raise ValueError("No tables found – page structure may have changed.")
table = tables[0]  # Rasterization table is first

rows = table.find_all('tr')
data = []

print(f"Found {len(rows)-1} potential GPU rows in the rasterization table.")  # Debug info

for row in rows[1:]:  # Skip header
    cols = row.find_all('td')
    if len(cols) < 8:  # Ensure full row (8 columns in raster table)
        continue
    
    gpu_name = cols[0].text.strip()
    
    # 1080p Ultra score – clean robustly (e.g., "100.0%" or "100.0% (197.5)")
    ultra_str = cols[4].text.strip()
    ultra_str = ultra_str.split('(')[0].strip()  # Remove FPS in parens if present
    ultra_str = ultra_str.rstrip('%').strip()
    try:
        perf_score = float(ultra_str)
    except ValueError:
        print(f"Skipping row due to invalid 1080p Ultra score: '{ultra_str}' in {gpu_name}")
        continue
    
    lowest_price = cols[1].text.strip()
    msrp = cols[2].text.strip()
    specs = cols[7].text.strip()  # Usually review links/notes
    
    # Simple manufacturer detection
    if 'GeForce' in gpu_name or 'RTX' in gpu_name or 'GTX' in gpu_name:
        manufacturer = 'NVIDIA'
    elif 'Radeon' in gpu_name or 'RX' in gpu_name:
        manufacturer = 'AMD'
    elif 'Arc' in gpu_name:
        manufacturer = 'Intel'
    else:
        manufacturer = 'Unknown'
    
    data.append({
        'gpu_name': gpu_name,
        'manufacturer': manufacturer,
        'perf_score': perf_score,  # Direct 1080p Ultra relative % – ideal for gaming
        'lowest_price': lowest_price,
        'msrp': msrp,
        'specs_notes': specs
    })

df_gpu = pd.DataFrame(data)
if df_gpu.empty:
    print("Warning: No GPUs were parsed – check page structure.")
else:
    df_gpu = df_gpu.sort_values('perf_score', ascending=False).reset_index(drop=True)

df_gpu.to_csv('gpu_benchmarks_2026.csv', index=False)
print(f"Successfully scraped {len(df_gpu)} GPUs!")
print("Top 10 GPUs:")
print(df_gpu[['gpu_name', 'perf_score', 'manufacturer']].head(10))

Found 34 potential GPU rows in the rasterization table.
Successfully scraped 34 GPUs!
Top 10 GPUs:
                    gpu_name  perf_score manufacturer
0           GeForce RTX 5090       100.0       NVIDIA
1           GeForce RTX 4090        95.2       NVIDIA
2           GeForce RTX 5080        84.9       NVIDIA
3     GeForce RTX 4080 Super        83.2       NVIDIA
4           GeForce RTX 4080        82.0       NVIDIA
5         Radeon RX 7900 XTX        79.4          AMD
6        GeForce RTX 5070 Ti        78.8       NVIDIA
7          Radeon RX 9070 XT        76.1          AMD
8  GeForce RTX 4070 Ti Super        74.1       NVIDIA
9          Radeon RX 7900 XT        73.3          AMD


In [8]:
# - Legacy CPUs & GPUs (with guaranteed low-end GPU additions)- 

headers = {'User-Agent': 'Mozilla/5.0'}

# --- Legacy CPU Scraper ---
url_cpu_legacy = "https://www.tomshardware.com/reviews/cpu-hierarchy,4312-2.html"
response = requests.get(url_cpu_legacy, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

tables = soup.find_all('table')
data_cpu_legacy = []

# Table 1: Recent legacy (e.g., 13900K down to i5-12400)
if len(tables) >= 1:
    table1 = tables[0]  # Usually first table
    rows1 = table1.find_all('tr')
    for row in rows1[1:]:  # Skip header
        cols = row.find_all('td')
        if len(cols) < 7: continue
        product = cols[0].text.strip()  # e.g., '$589 - Core i9-13900K'
        gaming_score_str = cols[1].text.strip().rstrip('%')  # 1080p
        try:
            perf_score = float(gaming_score_str) if gaming_score_str else 0.0
        except ValueError:
            continue
        arch = cols[3].text.strip()
        cores_threads = cols[4].text.strip()
        clocks = cols[5].text.strip()
        tdp = cols[6].text.strip()
        price = re.match(r'^\$(\d+)', product).group(1) if re.match(r'^\$(\d+)', product) else ''
        cpu_name = re.sub(r'^\$\d+ - ', '', product).strip()  # Extract name
        manufacturer = 'AMD' if 'Ryzen' in cpu_name else 'Intel'
        data_cpu_legacy.append({
            'cpu_name': cpu_name,
            'manufacturer': manufacturer,
            'perf_score': perf_score,
            'architecture': arch,
            'cores_threads': cores_threads,
            'base_boost_ghz': clocks,
            'tdp': tdp,
            'price': price
        })

# Table 2: Older legacy (e.g., 12900K down)
if len(tables) >= 2:
    table2 = tables[1]
    rows2 = table2.find_all('tr')
    for row in rows2[1:]:
        cols = row.find_all('td')
        if len(cols) < 7: continue
        product = cols[0].text.strip()  # e.g., 'Intel Core i9-12900K DDR4 / DDR5'
        gaming_score_str = cols[1].text.strip().split('/')[0].rstrip('%').strip()  # Take DDR4/high value
        try:
            perf_score = float(gaming_score_str) if gaming_score_str else 0.0
        except ValueError:
            continue
        cpu_name = re.sub(r' DDR4 / DDR5$', '', product).strip()
        arch = cols[3].text.strip()
        cores_threads = cols[4].text.strip()
        clocks = cols[5].text.strip()
        tdp = cols[6].text.strip()
        manufacturer = 'AMD' if 'Ryzen' in cpu_name or 'Threadripper' in cpu_name else 'Intel'
        data_cpu_legacy.append({
            'cpu_name': cpu_name,
            'manufacturer': manufacturer,
            'perf_score': perf_score,
            'architecture': arch,
            'cores_threads': cores_threads,
            'base_boost_ghz': clocks,
            'tdp': tdp,
            'price': ''  # No prices here
        })

df_cpu_legacy = pd.DataFrame(data_cpu_legacy)
df_cpu_legacy.drop_duplicates(subset=['cpu_name'], keep='first', inplace=True)

# Append to main CPU and save
df_cpu_current = pd.read_csv('cpu_benchmarks_2026.csv')
df_cpu_extended = pd.concat([df_cpu_current, df_cpu_legacy], ignore_index=True)
df_cpu_extended.to_csv('cpu_benchmarks_2026_extended.csv', index=False)
print(f"Extended CPUs: {len(df_cpu_extended)} (added {len(df_cpu_legacy)} legacy)")

# --- Legacy GPU Scrapers (Updated for Current Structure) ---

# Rasterization (Standard Gaming) + Ray Tracing Legacy: 4388-2.html
url_gpu_legacy_gaming = "https://www.tomshardware.com/reviews/gpu-hierarchy,4388-2.html"
response = requests.get(url_gpu_legacy_gaming, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

tables = soup.find_all('table')
data_gpu_legacy = []

print(f"Found {len(tables)} tables on legacy gaming GPU page.")

# Table 0: Rasterization (standard gaming – prioritize this)
if len(tables) >= 1:
    table_raster = tables[0]
    for row in table_raster.find_all('tr')[1:]:
        cols = row.find_all('td')
        if len(cols) < 6: continue
        gpu_name = cols[0].text.strip()
        # 1080p Ultra is now typically cols[1] (check page: Graphics Card | 1080p Ultra | 1080p Medium | ...)
        ultra_str = cols[1].text.strip().split('(')[0].rstrip('%').strip()  # e.g., "100.0"
        try:
            perf_score = float(ultra_str)
        except ValueError:
            continue
        specs = cols[5].text.strip() if len(cols) > 5 else ''
        arch = specs.split(',')[0].strip('[').strip() if specs else ''
        tdp = specs.split(',')[-1].strip().rstrip(']') if specs else ''
        manufacturer = 'NVIDIA' if 'GeForce' in gpu_name or 'RTX' in gpu_name else 'AMD' if 'Radeon' in gpu_name else 'Unknown'
        data_gpu_legacy.append({
            'gpu_name': gpu_name,
            'manufacturer': manufacturer,
            'perf_score': perf_score,
            'architecture': arch,
            'tdp': tdp,
            'source': 'raster'  # Mark priority
        })

# Table 1: Ray Tracing (fallback)
if len(tables) >= 2:
    table_rt = tables[1]
    for row in table_rt.find_all('tr')[1:]:
        cols = row.find_all('td')
        if len(cols) < 6: continue
        gpu_name = cols[0].text.strip()
        ultra_str = cols[2].text.strip().split('(')[0].rstrip('%').strip()  # 1080p Ultra usually cols[2]
        try:
            perf_score = float(ultra_str)
        except ValueError:
            continue
        specs = cols[5].text.strip() if len(cols) > 5 else ''
        arch = specs.split(',')[0].strip('[').strip() if specs else ''
        tdp = specs.split(',')[-1].strip().rstrip(']') if specs else ''
        manufacturer = 'NVIDIA' if 'GeForce' in gpu_name or 'RTX' in gpu_name else 'AMD' if 'Radeon' in gpu_name else 'Unknown'
        data_gpu_legacy.append({
            'gpu_name': gpu_name,
            'manufacturer': manufacturer,
            'perf_score': perf_score,
            'architecture': arch,
            'tdp': tdp,
            'source': 'rt'
        })

# Content Creation Legacy: 4388-3.html (lower priority fallback)
url_gpu_cc = "https://www.tomshardware.com/reviews/gpu-hierarchy,4388-3.html"
response = requests.get(url_gpu_cc, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

tables_cc = soup.find_all('table')
if tables_cc:
    table_cc = tables_cc[0]  # Main 2020-2021 table
    for row in table_cc.find_all('tr')[1:]:
        cols = row.find_all('td')
        if len(cols) < 6: continue
        gpu_name = cols[2].text.strip()  # GPU name is often cols[2] due to header layout
        score_str = cols[1].text.strip().rstrip('%')
        try:
            perf_score = float(score_str)
        except ValueError:
            continue
        base_boost = cols[3].text.strip()
        arch = base_boost.split()[0] if base_boost else ''
        tdp = cols[5].text.strip()
        manufacturer = 'NVIDIA' if 'Nvidia' in gpu_name or 'GeForce' in gpu_name else 'AMD' if 'AMD' in gpu_name else 'Unknown'
        data_gpu_legacy.append({
            'gpu_name': gpu_name,
            'manufacturer': manufacturer,
            'perf_score': perf_score,
            'architecture': arch,
            'tdp': tdp,
            'source': 'cc'
        })

df_gpu_legacy = pd.DataFrame(data_gpu_legacy)

# Prioritize: raster > rt > cc
priority = {'raster': 0, 'rt': 1, 'cc': 2}
df_gpu_legacy['priority'] = df_gpu_legacy['source'].map(priority)
df_gpu_legacy = df_gpu_legacy.sort_values('priority').drop_duplicates(subset=['gpu_name'], keep='first').drop(columns=['source', 'priority'])

# === 10 Manually Gathered Low-End GPUS ===
manual_low_end_gpus = [
    {"gpu_name": "GeForce GTX 750 Ti", "manufacturer": "NVIDIA", "perf_score": 20, "lowest_price": "used ~$50", "msrp": "N/A", "specs_notes": "Maxwell 2GB", "architecture": "Maxwell", "tdp": "60W"},
    {"gpu_name": "GeForce GT 1030", "manufacturer": "NVIDIA", "perf_score": 18, "lowest_price": "used ~$80", "msrp": "N/A", "specs_notes": "Pascal 2GB", "architecture": "Pascal", "tdp": "30W"},
    {"gpu_name": "GeForce GTX 660 Ti", "manufacturer": "NVIDIA", "perf_score": 25, "lowest_price": "used ~$60", "msrp": "N/A", "specs_notes": "Kepler 2GB", "architecture": "Kepler", "tdp": "150W"},
    {"gpu_name": "GeForce GTX 970", "manufacturer": "NVIDIA", "perf_score": 48, "lowest_price": "used ~$120", "msrp": "N/A", "specs_notes": "Maxwell 4GB", "architecture": "Maxwell", "tdp": "145W"},
    {"gpu_name": "GeForce GTX 1650", "manufacturer": "NVIDIA", "perf_score": 38, "lowest_price": "used ~$150", "msrp": "N/A", "specs_notes": "Turing 4GB", "architecture": "Turing", "tdp": "75W"},
    {"gpu_name": "Radeon HD 7850", "manufacturer": "AMD", "perf_score": 22, "lowest_price": "used ~$40", "msrp": "N/A", "specs_notes": "GCN 2GB", "architecture": "GCN", "tdp": "130W"},
    {"gpu_name": "Radeon R7 260X", "manufacturer": "AMD", "perf_score": 28, "lowest_price": "used ~$50", "msrp": "N/A", "specs_notes": "GCN 2GB", "architecture": "GCN", "tdp": "115W"},
    {"gpu_name": "Radeon RX 550", "manufacturer": "AMD", "perf_score": 32, "lowest_price": "used ~$70", "msrp": "N/A", "specs_notes": "GCN 4GB", "architecture": "Polaris", "tdp": "50W"},
    {"gpu_name": "GeForce 8800 GT", "manufacturer": "NVIDIA", "perf_score": 12, "lowest_price": "used ~$30", "msrp": "N/A", "specs_notes": "512MB", "architecture": "G92", "tdp": "105W"},
    {"gpu_name": "Radeon HD 6950", "manufacturer": "AMD", "perf_score": 25, "lowest_price": "used ~$45", "msrp": "N/A", "specs_notes": "Cayman 2GB", "architecture": "Cayman", "tdp": "200W"}
]

df_manual = pd.DataFrame(manual_low_end_gpus)

print(f"Added {len(df_manual)} manually curated low-end GPUs")

# Combine: scraped legacy + manual entries
df_gpu_legacy_all = pd.concat([df_gpu_legacy, df_manual], ignore_index=True)

# Drop duplicates: manual entries win if name matches (they come last, so keep='last')
df_gpu_legacy_all.drop_duplicates(subset=['gpu_name'], keep='last', inplace=True)

# Load current (modern) GPUs
df_gpu_current = pd.read_csv('gpu_benchmarks_2026.csv')

# Final extended list: current + all legacy (including manual)
df_gpu_extended = pd.concat([df_gpu_current, df_gpu_legacy_all], ignore_index=True)
df_gpu_extended.drop_duplicates(subset=['gpu_name'], keep='first', inplace=True)  # Current always wins over legacy

# Save
df_gpu_extended.to_csv('gpu_benchmarks_2026_extended.csv', index=False)

print(f"\nFinal Extended GPUs: {len(df_gpu_extended)} total")
print(f"  - From current page: {len(df_gpu_current)}")
print(f"  - From legacy scrape: {len(df_gpu_legacy)}")
print(f"  - From manual additions: {len(df_manual)}")
print(f"  - After deduplication: {len(df_gpu_extended)}")

# Show the manual ones made it
print("\nVerification — your 10 manual low-end cards are present:")
print(df_gpu_extended[df_gpu_extended['gpu_name'].isin([row['gpu_name'] for row in manual_low_end_gpus])][['gpu_name', 'perf_score', 'manufacturer']].to_string(index=False))

Extended CPUs: 104 (added 77 legacy)
Found 2 tables on legacy gaming GPU page.
Extended GPUs: 70 (added 36 legacy models)
Top legacy additions:
   gpu_name  perf_score
0     GA102       100.0
2   Navi 21        97.0
6     GA104        81.5
7     TU102        79.5
10  Navi 22        73.3
12    GV100        68.7
13    TU104        66.8
15    GP102        61.1
17  Vega 20        58.9
19  Navi 23        57.7


In [7]:
# - Steam Application Data - 

zip_url = "https://zenodo.org/records/17266923/files/steam_dataset_2025_csv_package_v1.zip"
zip_path = "steam_dataset_2025_csv_package_v1.zip"
extract_dir = "steam_data_2025"

# Download if not present
if not os.path.exists(zip_path):
    print(f"Downloading Steam dataset from Zenodo (~300 MB)...")
    print("Source: https://zenodo.org/records/17266923")
    
    response = requests.get(zip_url, stream=True)
    response.raise_for_status()  # Raise error if download fails
    
    total_size = int(response.headers.get('content-length', 0))
    block_size = 1024 * 1024  # 1 MB chunks
    
    with open(zip_path, 'wb') as f, tqdm(
        total=total_size,
        unit='iMB',
        unit_scale=True,
        desc=zip_path
    ) as pbar:
        for data in response.iter_content(block_size):
            f.write(data)
            pbar.update(len(data))
    
    print(f"Download complete: {zip_path}")
else:
    print(f"Found existing ZIP: {zip_path}")

# Extract
if not os.path.exists(extract_dir):
    print(f"Extracting {zip_path} to {extract_dir}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"Extracted to {extract_dir}")
else:
    print(f"Already extracted to {extract_dir}")

# List contents for confirmation
print("\nExtracted files:")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    print(zip_ref.namelist())

Source: https://zenodo.org/records/17266923


steam_dataset_2025_csv_package_v1.zip: 100%|██████████| 320M/320M [02:21<00:00, 2.26MiMB/s] 


Download complete: steam_dataset_2025_csv_package_v1.zip
Extracting steam_dataset_2025_csv_package_v1.zip to steam_data_2025...
Extracted to steam_data_2025

Extracted files:
['steam_dataset_2025_csv/application_categories.csv', 'steam_dataset_2025_csv/application_developers.csv', 'steam_dataset_2025_csv/application_genres.csv', 'steam_dataset_2025_csv/application_platforms.csv', 'steam_dataset_2025_csv/application_publishers.csv', 'steam_dataset_2025_csv/applications.csv', 'steam_dataset_2025_csv/categories.csv', 'steam_dataset_2025_csv/developers.csv', 'steam_dataset_2025_csv/genres.csv', 'steam_dataset_2025_csv/platforms.csv', 'steam_dataset_2025_csv/publishers.csv', 'steam_dataset_2025_csv/reviews.csv', 'steam_dataset_2025_csv/MANIFEST.json']


In [8]:

# Path to the extracted CSV

csv_path = 'steam_data_2025/steam_dataset_2025_csv/applications.csv' 

df = pd.read_csv(csv_path)

print(f"Original shape: {df.shape}")
print(f"Unique types: {df['type'].unique()}")

# Step 1: Keep only games
df = df[df['type'] == 'game'].copy()
print(f"After keeping games: {df.shape}")

# Step 2: Keep rows with at least one minimum requirement
req_cols_min = ['mat_pc_processor_min', 'mat_pc_graphics_min', 'mat_pc_memory_min']
df = df.dropna(subset=req_cols_min, how='all')  # Drop if ALL three are NaN
print(f"After requiring at least one min req: {df.shape}")

# Step 3: Select only needed columns
columns_to_keep = [
    'appid',
    'name',
    'release_date',
    'is_free',
    'required_age',
    'mat_supports_windows',  # Useful filter
    'recommendations_total',  # Proxy for popularity
    'mat_pc_os_min',
    'mat_pc_processor_min',
    'mat_pc_memory_min',
    'mat_pc_graphics_min',
    'mat_pc_os_rec',
    'mat_pc_processor_rec',
    'mat_pc_memory_rec',
    'mat_pc_graphics_rec'
]

df_slim = df[columns_to_keep].copy()

# Step 4: Optional - further reduce size (recommended for review)
# Keep top 50,000 most popular (by recommendations) or most recent
if len(df_slim) > 50000:
    # Prioritize games with reviews first, then recent
    df_slim = df_slim.sort_values(
        by=['recommendations_total', 'release_date'],
        ascending=[False, False]
    ).head(50000)
    print(f"Reduced to top 50K by popularity + recency: {df_slim.shape}")

# Step 5: Clean memory field (extract number in GB)
def extract_ram_gb(text):
    if pd.isna(text):
        return None
    match = re.search(r'(\d+)\s*(GB|MB)', str(text), re.IGNORECASE)
    if match:
        val = int(match.group(1))
        unit = match.group(2).upper()
        return val / 1024 if unit == 'MB' else val
    return None

df_slim['min_ram_gb'] = df_slim['mat_pc_memory_min'].apply(extract_ram_gb)
df_slim['rec_ram_gb'] = df_slim['mat_pc_memory_rec'].apply(extract_ram_gb)

# Final save
output_path = 'steam_games_slim.csv'
df_slim.to_csv(output_path, index=False)

file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"\nSlim dataset saved: {output_path}")
print(f"Final shape: {df_slim.shape}")
print(f"File size: {file_size_mb:.1f} MB")
print("\nColumns in slim version:")
print(df_slim.columns.tolist())

C:\Users\Anarchial_Order\AppData\Local\Temp\ipykernel_28156\2134443908.py:5: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


Original shape: (239664, 30)
Unique types: ['game' 'demo' 'dlc' nan 'music' 'video']
After keeping games: (150279, 30)
After requiring at least one min req: (137808, 30)
Reduced to top 50K by popularity + recency: (50000, 15)

Slim dataset saved: steam_games_slim.csv
Final shape: (50000, 17)
File size: 9.4 MB

Columns in slim version:
['appid', 'name', 'release_date', 'is_free', 'required_age', 'mat_supports_windows', 'recommendations_total', 'mat_pc_os_min', 'mat_pc_processor_min', 'mat_pc_memory_min', 'mat_pc_graphics_min', 'mat_pc_os_rec', 'mat_pc_processor_rec', 'mat_pc_memory_rec', 'mat_pc_graphics_rec', 'min_ram_gb', 'rec_ram_gb']
